# Surrogate gradients

The code of [sparxml.dev/learn/surrogate-gradients](https://sparxml.dev/learn/surrogate-gradients/), run on a CPU runtime. The first cell installs sparx.

In [ ]:
%pip install -q 'sparxml[nir] @ git+https://github.com/AshishKumar4/sparx'

In [ ]:
import jax
import jax.numpy as jnp

from sparx.surrogate import ATan, FastSigmoid, spike

v = jnp.array([-1.0, -0.1, 0.0, 0.1, 1.0])       # membrane - threshold
print(spike(v, ATan()))                           # [0. 0. 1. 1. 1.]

# The forward pass is the step; the gradient is the surrogate's.
for surrogate in (ATan(), FastSigmoid(25.0)):
    slope = jax.vmap(jax.grad(lambda x, s=surrogate: spike(x, s)))(v)
    print(type(surrogate).__name__, slope)

In [ ]:
import jax
import jax.numpy as jnp
import optax

import sparx
from sparx.dynamics import LIFCell, decay
from sparx.surrogate import ATan

T, N = 200, 40
inputs = jax.random.uniform(jax.random.key(0), (T, N)) < 0.04
trains = inputs.astype(jnp.float32)
# Fire at these steps
target = jnp.zeros(T).at[jnp.array([40, 90, 150])].set(1.0)
cell = LIFCell(decay=decay(tau=10.0), threshold=1.0,
               surrogate=ATan())
keep = decay(tau=10.0)              # an exponential filter of 10 steps


def smooth(spikes):
    def step(f, s):
        f = keep * f + s
        return f, f
    return jax.lax.scan(step, 0.0, spikes)[1]


def loss(w):
    out, _ = sparx.run(cell, trains @ w)     # spikes exactly 0 or 1
    return jnp.mean((smooth(out.value) - smooth(target)) ** 2)


w = 0.35 * jax.random.normal(jax.random.key(1), (N,))
adam = optax.adam(0.04)
state = adam.init(w)
grad = jax.jit(jax.grad(loss))       # the slope comes from ATan
for _ in range(400):
    updates, state = adam.update(grad(w), state)
    w = optax.apply_updates(w, updates)
fired = sparx.run(cell, trains @ w)[0].value
print("fires at", jnp.flatnonzero(fired))